# RÉ-EXÉCUTION v1 — EfficientNet-B2

> ⚠ **Ceci n'est PAS le notebook d'origine** (supprimé). C'est une **ré-exécution** du code `code/v1/` du dépôt, faite le jour de l'export (voir la date des sorties), avec les hyperparamètres d'origine sauvegardés (`models/.../Hyperparametres/`) et **sans nouvelle recherche d'hyperparamètres**. Les scores peuvent différer légèrement de ceux des fichiers `models/.../v1/Resultats/` (TensorFlow/CUDA non déterministes).

Avant d'exécuter : *Add Input* → `daikhiyassine/dataset` **avant de démarrer la session**, accélérateur GPU, Internet activé. Ensuite *Save Version → Save & Run All*, puis *File → Download notebook*.

In [2]:
!pip install -q keras-tuner

In [3]:
!nvidia-smi
!git clone https://github.com/Yacindaikhi/A-Cross-Architecture-Benchmark-of-DL-Models-for-Structural-Defect-Detection-on-Uncurated-Data.git repo
!git -C repo log -1 --format='%h %ad %s'

Tue Nov 24 08:28:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 1. Nettoyage + partition tel qu'exécuté pour B2 (MD5, fuites inter-splits, RGB ; SANS redimensionnement)
!CLEAN_PATH=/kaggle/working/dataset_clean_b2 python repo/code/v1/data/02_clean_dataset_b2_fullres.py

Étape 1/3 : identification des doublons à retirer...
   ❌ Retiré : test/spalling/427_yang_jpg.rf.642f33876b4e27774adef8e4bddd1afa.jpg (doublon de train/cracks/Image_-192-_jpg.rf.825feee6b7b43e6279e68bc3c6222db3.jpg)
   ❌ Retiré : val/spalling/train_train_0318.jpg (doublon de train/cracks/train_train_0318.jpg)
   ❌ Retiré : test/cracks/Image_-120-_jpg.rf.4ed26783423892fd1ef5ee730e985a50.jpg (doublon de val/spalling/262_yang_jpg.rf.2990160898519c5b450e727574fe50f5.jpg)

Total fichiers à exclure : 3

Étape 2/3 : copie + conversion RGB...

Images copiées : 21010
Images converties en RGB : 626

Étape 3/3 : vérification finale...

Comptage final (dataset nettoyé) :

TRAIN (total: 14707)
   corrosion: 3936
   spalling: 3747
   normal: 3594
   cracks: 3430

VAL (total: 3151)
   corrosion: 844
   spalling: 802
   normal: 770
   cracks: 735

TEST (total: 3152)
   corrosion: 844
   spalling: 803
   normal: 771
   cracks: 734

✅ Dataset nettoyé disponible dans : /kaggle/working/dataset_clean_b2
✅ 

In [5]:
# Garde-fou : arrête l'exécution si le nettoyage a échoué (évite des heures d'entraînement inutiles)
import os
root='/kaggle/working/dataset_clean_b2'
n={sp:sum(len(os.listdir(os.path.join(root,sp,c))) for c in os.listdir(os.path.join(root,sp))) for sp in ('train','val','test')}
print(n)
assert n=={'train': 14707, 'val': 3151, 'test': 3152}, 'effectifs inattendus : ' + str(n) + ' (attendu 14707 / 3151 / 3152)'

{'train': 14707, 'val': 3151, 'test': 3152}


In [6]:
# 2. Manifeste de partition (attendu : train=14707 val=3151 test=3152)
!python repo/tools/make_manifest.py /kaggle/working/dataset_clean_b2 /kaggle/working/v1_clean_b2_manifest.csv

21010 images -> /kaggle/working/v1_clean_b2_manifest.csv | train=14707 val=3151 test=3152
par classe/partition: {('train', 'corrosion'): 3936, ('train', 'cracks'): 3430, ('train', 'normal'): 3594, ('train', 'spalling'): 3747, ('val', 'corrosion'): 844, ('val', 'cracks'): 735, ('val', 'normal'): 770, ('val', 'spalling'): 802, ('test', 'corrosion'): 844, ('test', 'cracks'): 734, ('test', 'normal'): 771, ('test', 'spalling'): 803}


In [7]:
# 3. Hyperparamètres d'origine (pas de nouvelle recherche)
!cp repo/models/efficientnet_b2/v1/Hyperparametres/best_hyperparameters.json /kaggle/working/best_hyperparameters_b2.json
!cat /kaggle/working/best_hyperparameters_b2.json

{
  "dense_units": 1024,
  "dropout_rate": 0.3,
  "learning_rate": 0.001,
  "unfreeze_layers": 50,
  "use_batchnorm": false,
  "optimizer": "rmsprop"
}

In [8]:
# 4. Entraînement
!python repo/code/v1/efficientnet_b2/train.py

Hyperparamètres utilisés : {'dense_units': 1024, 'dropout_rate': 0.3, 'learning_rate': 0.001, 'unfreeze_layers': 50, 'use_batchnorm': False, 'optimizer': 'rmsprop'}
Found 14707 files belonging to 4 classes.
Found 3151 files belonging to 4 classes.
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model: "functional_1"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 260, 260,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────

In [9]:
# 5. Évaluation sur le split test
!python repo/code/v1/efficientnet_b2/test.py

Found 3152 files belonging to 4 classes.
E0000 00:00:1795514879.813208    1540 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795514879.996287    1540 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795514882.019370    1540 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795514882.189213    1540 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1795514913.870350    1541 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, pl

In [10]:
# 6. Environnement figé + empreintes des fichiers produits
!python repo/tools/capture_env.py kaggle_v1_efficientnet_b2_rerun
!python repo/tools/hash_files.py /kaggle/working/aigis_efficientnetb2_final.keras
!cat environments/kaggle_v1_efficientnet_b2_rerun_env.json

{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "nvidia_smi": "Tesla T4, 580.178.04, 15360 MiB\nTesla T4, 580.178.04, 15360 MiB",
  "nvcc": [
    "Build cuda_13.0.r13.0/compiler.36424714_0"
  ],
  "tensorflow": "2.21.0",
  "keras": "3.13.2",
  "keras_tuner": "1.4.8",
  "torch": "2.11.0+cu130",
  "torch_cuda": "13.0",
  "torchvision": "0.26.0+cu130",
  "ultralytics": null,
  "sklearn": "1.6.1",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "PIL": "13.0.0"
}
| `aigis_efficientnetb2_final.keras` | 60586848 | `52c91d304a0811bd2f4861d175ad20664874dcef124b0bb6f3458682463af272` |
{
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "nvidia_smi": "Tesla T4, 580.178.04, 15360 MiB\nTesla T4, 580.178.04, 15360 MiB",
  "nvcc": [
    "Build cuda_13.0.r13.0/compiler.36424714_0"
  ],
  "tensorflow": "2.21.0",
  "keras": "3.13.2",
  "keras_tuner": "1.4.8",
  "

In [1]:
!ls /kaggle/input/datasets/daikhiyassine/dataset

corrosion  cracks  normal  spalling  test  train  val
